In [135]:
import numpy as np
import open3d as o3d
from plyfile import PlyData 
from pycpd import DeformableRegistration
from scipy.spatial import cKDTree
import smplx
import torch
import trimesh


Step 1. Read both meshes in and convert to point clouds.

pixie_mesh = Mesh from PIXIE model
alpha_mesh = Mesh derived geometrically from depth data
 

In [136]:
# read in the meshes
alpha_mesh = o3d.io.read_triangle_mesh("/Users/adeleyounis/Desktop/Capstone/wAI/3D-processing/alpha_mesh.obj")
alpha_mesh.compute_vertex_normals()

pixie_mesh = o3d.io.read_triangle_mesh("/Users/adeleyounis/Desktop/Capstone/wAI/3D-processing/modelling/output/adele/adele.obj")
pixie_mesh.compute_vertex_normals()

alpha_mesh.paint_uniform_color([0.7, 0.7, 0.7])
pixie_mesh.paint_uniform_color([1.0, 0.2, 0.2])

# verify that they are triangular meshes with points and triangles
print(f"alpha_mesh: {alpha_mesh}")
print(f"pixie_mesh: {pixie_mesh}")

# o3d.visualization.draw_geometries(
#     [alpha_mesh, pixie_mesh],
#     mesh_show_back_face=True
# )

alpha_mesh: TriangleMesh with 8154 points and 17132 triangles.
pixie_mesh: TriangleMesh with 11313 points and 20908 triangles.


In [138]:
# refit to smplx pose
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = smplx.create(
    model_path="/Users/adeleyounis/Desktop/Capstone/wAI/3D-processing/modelling/models/",
    model_type="smplx",
    gender="neutral",
    use_pca=False
).to(device)

print(model)
print("num_betas:", model.num_betas)
print("num_joints:", model.J_regressor.shape[0])
print("shapedirs:", model.shapedirs.shape)


verts = torch.tensor(np.asarray(pixie_mesh.vertices), dtype=torch.float32).to(device)

global_orient = torch.zeros((1, 3), requires_grad=True, device=device)
body_pose    = torch.zeros((1, 63), requires_grad=True, device=device)

left_hand    = torch.zeros((1, 45), requires_grad=True, device=device)
right_hand   = torch.zeros((1, 45), requires_grad=True, device=device)

jaw_pose     = torch.zeros((1, 3), requires_grad=True, device=device)
leye_pose    = torch.zeros((1, 3), requires_grad=True, device=device)
reye_pose    = torch.zeros((1, 3), requires_grad=True, device=device)

betas = torch.zeros((1, 16), requires_grad=True, device=device)
trans = torch.zeros((1, 3), requires_grad=True, device=device)

optimizer = torch.optim.Adam(
    [
        global_orient,
        body_pose,
        left_hand,
        right_hand,
        jaw_pose,
        leye_pose,
        reye_pose,
        betas,
        trans
    ],
    lr=1e-2
)


for _ in range(300):
    output = model(
        global_orient=global_orient,
        body_pose=body_pose,
        left_hand_pose=left_hand,
        right_hand_pose=right_hand,
        jaw_pose=jaw_pose,
        leye_pose=leye_pose,
        reye_pose=reye_pose,
        betas=betas,
        transl=trans
    )

def chamfer_distance(a, b, chunk_size=2048):
    """
    a: (Na,3) SMPL-X vertices
    b: (Nb,3) PIXIE vertices
    """
    a = a.unsqueeze(0)
    b = b.unsqueeze(0)

    Na = a.shape[1]
    Nb = b.shape[1]

    dist_ab = []
    for i in range(0, Na, chunk_size):
        aa = a[:, i:i+chunk_size]
        d = torch.cdist(aa, b)
        dist_ab.append(d.min(dim=2)[0])
    dist_ab = torch.cat(dist_ab, dim=1)

    dist_ba = []
    for i in range(0, Nb, chunk_size):
        bb = b[:, i:i+chunk_size]
        d = torch.cdist(bb, a)
        dist_ba.append(d.min(dim=2)[0])
    dist_ba = torch.cat(dist_ba, dim=1)

    return dist_ab.mean() + dist_ba.mean()


for it in range(500):
    optimizer.zero_grad()

    output = model(
        global_orient=global_orient,
        body_pose=body_pose,
        left_hand_pose=left_hand,
        right_hand_pose=right_hand,
        jaw_pose=jaw_pose,
        leye_pose=leye_pose,
        reye_pose=reye_pose,
        betas=betas,
        transl=trans
    )

    loss_data = chamfer_distance(output.vertices[0], verts)

    pose_reg = 1e-4 * (
        body_pose.pow(2).mean() +
        left_hand.pow(2).mean() +
        right_hand.pow(2).mean()
    )

    loss = loss_data + pose_reg
    loss.backward()
    optimizer.step()

    if it % 25 == 0:
        print(f"iter {it:03d} | loss {loss.item():.6f}")

# -------------------------
# Extract joints (RESULT)
# -------------------------
joints = output.joints[0].detach().cpu().numpy()  # (55,3)
print("Recovered joints shape:", joints.shape)

# -------------------------
# Optional: visualize fitted SMPL-X mesh
# -------------------------
smpl_mesh = o3d.geometry.TriangleMesh()
smpl_mesh.vertices = o3d.utility.Vector3dVector(
    output.vertices[0].detach().cpu().numpy()
)
smpl_mesh.triangles = o3d.utility.Vector3iVector(model.faces)

o3d.visualization.draw_geometries(
    [pixie_mesh, smpl_mesh],
    mesh_show_back_face=True
)



SMPLX(
  Gender: NEUTRAL
  Number of joints: 55
  Betas: 16
  Flat hand mean: False
  Number of Expression Coefficients: 10
  (vertex_joint_selector): VertexJointSelector()
)
num_betas: 16
num_joints: 55
shapedirs: torch.Size([10475, 3, 16])
iter 000 | loss 0.426602
iter 025 | loss 0.123258
iter 050 | loss 0.074140
iter 075 | loss 0.067326
iter 100 | loss 0.063579
iter 125 | loss 0.060989
iter 150 | loss 0.058310
iter 175 | loss 0.056399
iter 200 | loss 0.054909
iter 225 | loss 0.053762
iter 250 | loss 0.053003
iter 275 | loss 0.052518
iter 300 | loss 0.052101
iter 325 | loss 0.051729
iter 350 | loss 0.051398
iter 375 | loss 0.051058
iter 400 | loss 0.050760
iter 425 | loss 0.050428
iter 450 | loss 0.050141
iter 475 | loss 0.049883
Recovered joints shape: (127, 3)


In [139]:
# scale and orient since pixie is scaled down
depth_size = np.array(alpha_mesh.get_max_bound() - alpha_mesh.get_min_bound())
pixie_size = np.array(pixie_mesh.get_max_bound() - pixie_mesh.get_min_bound())

scale =  pixie_size.max() / depth_size.max()
print(scale)

alpha_mesh.scale(scale, center=(0,0,0))

# Flip 180° around X (fix upside down)
pixie_mesh.rotate(pixie_mesh.get_rotation_matrix_from_xyz((np.pi, 0, 0)), center=(0,0,0))

# Rotate 180° around Y (face the camera)
pixie_mesh.rotate(pixie_mesh.get_rotation_matrix_from_xyz((0, np.pi, 0)), center=(0,0,0))

depth_center = alpha_mesh.get_center()
pixie_center = pixie_mesh.get_center()

pixie_mesh.translate(depth_center - pixie_center)

# o3d.visualization.draw_geometries(
#     [alpha_mesh, pixie_mesh],
#     mesh_show_back_face=True
# )

0.30834542822315186


TriangleMesh with 11313 points and 20908 triangles.

In [140]:
# get head from pixie model - delete alpha model head
min_bound = alpha_mesh.get_min_bound()
max_bound = alpha_mesh.get_max_bound()

print(min_bound, max_bound)

verts = np.asarray(alpha_mesh.vertices)
y_min, y_max = min_bound[1], max_bound[1]
x_min, x_max = min_bound[0], max_bound[0]
x_mean = (x_min + x_max) / 2

# Depth arms = points far left or far right
# (tighter threshold = fewer points kept)
arm_threshold = 0.25 * (x_max - x_min)

left_arm_mask  = np.asarray(alpha_mesh.vertices)[:,0] < (x_mean - arm_threshold)
right_arm_mask = np.asarray(alpha_mesh.vertices)[:,0] > (x_mean + arm_threshold)

depth_arm_mask = left_arm_mask | right_arm_mask
head_mask = verts[:,1] > (y_min + 0.850 * (y_max - y_min))  # top 15%


[-0.35374313 -0.71677978  2.33917001] [0.41443783 0.80971198 3.12563588]


In [141]:
# convert to point cloud
alpha_pcd = alpha_mesh.sample_points_uniformly(number_of_points=50000)
pixie_pcd = pixie_mesh.sample_points_uniformly(number_of_points=50000)

alpha_pcd.estimate_normals()
pixie_pcd.estimate_normals()

print(len(alpha_pcd.points))
print(len(pixie_pcd.points))

50000
50000


In [142]:
def preprocess(pcd, voxel=0.02):
    p = pcd.voxel_down_sample(voxel)
    p.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius=voxel*2, max_nn=30))
    f = o3d.pipelines.registration.compute_fpfh_feature(
        p, o3d.geometry.KDTreeSearchParamHybrid(radius=voxel*5, max_nn=100)
    )
    return p, f

def part_icp(source_pcd, target_pcd, voxel=0.015, max_corr_dist_init=0.08):
    source_down, source_f = preprocess(source_pcd, voxel)
    target_down, target_f = preprocess(target_pcd, voxel)

    # Global registration (RANSAC)
    result_ransac = o3d.pipelines.registration.registration_ransac_based_on_feature_matching(
        source_down, target_down, source_f, target_f,
        mutual_filter=False,
        max_correspondence_distance=max_corr_dist_init,
        estimation_method=o3d.pipelines.registration.TransformationEstimationPointToPoint(False),
        ransac_n=4,
        checkers=[
            o3d.pipelines.registration.CorrespondenceCheckerBasedOnEdgeLength(0.8),
            o3d.pipelines.registration.CorrespondenceCheckerBasedOnDistance(max_corr_dist_init)
        ],
        criteria=o3d.pipelines.registration.RANSACConvergenceCriteria(max_iteration=200000, confidence=0.999)
    )

    # ICP refinement (multi-step, after RANSAC)
    result_icp_1 = o3d.pipelines.registration.registration_icp(
        source_pcd, target_pcd,
        max_correspondence_distance=max_corr_dist_init,
        init=result_ransac.transformation,
        estimation_method=o3d.pipelines.registration.TransformationEstimationPointToPlane(),
        criteria=o3d.pipelines.registration.ICPConvergenceCriteria(max_iteration=100)
    )

    result_icp_2 = o3d.pipelines.registration.registration_icp(
        source_pcd, target_pcd,
        max_correspondence_distance=max_corr_dist_init/2,
        init=result_icp_1.transformation,
        estimation_method=o3d.pipelines.registration.TransformationEstimationPointToPlane(),
        criteria=o3d.pipelines.registration.ICPConvergenceCriteria(max_iteration=200)
    )

    result_icp_3 = o3d.pipelines.registration.registration_icp(
        source_pcd, target_pcd,
        max_correspondence_distance=max_corr_dist_init/5,
        init=result_icp_2.transformation,
        estimation_method=o3d.pipelines.registration.TransformationEstimationPointToPlane(),
        criteria=o3d.pipelines.registration.ICPConvergenceCriteria(max_iteration=300)
    )

    return result_icp_3.transformation

In [150]:
def split_mesh_three_parts(mesh, top_ratio=0.4, mid_ratio=0.3):
    """
    Split mesh into three parts along the Y axis:
    - top: top_ratio fraction of height
    - mid: mid_ratio fraction of height
    - bottom: remaining fraction
    """
    V = np.asarray(mesh.vertices)
    y_min, y_max = V[:,1].min(), V[:,1].max()
    height = y_max - y_min

    split_y1 = y_min + top_ratio * height
    split_y2 = y_min + (top_ratio + mid_ratio) * height

    # Masks
    top_mask    = V[:,1] >= split_y2
    mid_mask    = (V[:,1] >= split_y1) & (V[:,1] < split_y2)
    bottom_mask = V[:,1] < split_y1

    top_mesh    = mesh.select_by_index(np.where(top_mask)[0], cleanup=True)
    mid_mesh    = mesh.select_by_index(np.where(mid_mask)[0], cleanup=True)
    bottom_mesh = mesh.select_by_index(np.where(bottom_mask)[0], cleanup=True)

    return top_mesh, mid_mesh, bottom_mesh


# Split Pixie and Alpha into 3 segments
pixie_top, pixie_mid, pixie_bottom = split_mesh_three_parts(pixie_mesh, top_ratio=0.3, mid_ratio=0.3)
alpha_top, alpha_mid, alpha_bottom = split_mesh_three_parts(alpha_mesh, top_ratio=0.3, mid_ratio=0.3)

# Sample points
def mesh_to_pcd(mesh, n_points=30000):
    return mesh.sample_points_poisson_disk(n_points)

pixie_top_pcd    = mesh_to_pcd(pixie_top)
pixie_mid_pcd    = mesh_to_pcd(pixie_mid)
pixie_bottom_pcd = mesh_to_pcd(pixie_bottom)

alpha_top_pcd    = mesh_to_pcd(alpha_top)
alpha_mid_pcd    = mesh_to_pcd(alpha_mid)
alpha_bottom_pcd = mesh_to_pcd(alpha_bottom)

# Register each segment separately
trans_top    = part_icp(pixie_top_pcd, alpha_top_pcd)
trans_mid    = part_icp(pixie_mid_pcd, alpha_mid_pcd)
trans_bottom = part_icp(pixie_bottom_pcd, alpha_bottom_pcd)

# Apply transformations to original meshes
pixie_top.transform(trans_top)
pixie_mid.transform(trans_mid)
pixie_bottom.transform(trans_bottom)

# Merge all 3 segments back together
pixie_combined = pixie_top + pixie_mid + pixie_bottom

o3d.visualization.draw_geometries(
    [alpha_mesh, pixie_combined],
    mesh_show_back_face=True
)


[Open3D WARNING] [RemoveDuplicatedTriangles] This mesh contains triangle uvs that are not handled in this function
[Open3D WARNING] [RemoveDegenerateTriangles] This mesh contains triangle uvs that are not handled in this function
[Open3D WARNING] [RemoveDuplicatedTriangles] This mesh contains triangle uvs that are not handled in this function
[Open3D WARNING] [RemoveDegenerateTriangles] This mesh contains triangle uvs that are not handled in this function
[Open3D WARNING] [RemoveDuplicatedTriangles] This mesh contains triangle uvs that are not handled in this function
[Open3D WARNING] [RemoveDegenerateTriangles] This mesh contains triangle uvs that are not handled in this function


In [143]:
# global rigid registration with RANSAC
# RANSAC samples downsampled point clouds and uses FPFH (fast point feature histograms) features

def split_mesh_top_bottom(mesh, split_ratio=0.45):
    """
    Split mesh into top half and bottom half along Y axis.
    split_ratio: fraction of height for bottom half (legs)
    """
    V = np.asarray(mesh.vertices)
    y_min, y_max = V[:,1].min(), V[:,1].max()
    split_y = y_min + split_ratio * (y_max - y_min)

    # bottom vertices mask
    bottom_mask = V[:,1] < split_y
    # top vertices mask
    top_mask = V[:,1] >= split_y

    top_mesh = mesh.select_by_index(np.where(top_mask)[0], cleanup=True)
    bottom_mesh = mesh.select_by_index(np.where(bottom_mask)[0], cleanup=True)

    return top_mesh, bottom_mesh

def preprocess(pcd, voxel=0.02):
    p = pcd.voxel_down_sample(voxel)
    p.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius=voxel*2, max_nn=30))
    f = o3d.pipelines.registration.compute_fpfh_feature(
        p, o3d.geometry.KDTreeSearchParamHybrid(radius=voxel*5, max_nn=100)
    )
    return p, f

# Split meshes
pixie_top, pixie_bottom = split_mesh_top_bottom(pixie_mesh, split_ratio=0.5)
alpha_top, alpha_bottom = split_mesh_top_bottom(alpha_mesh, split_ratio=0.5)

def mesh_to_pcd(mesh, n_points=30000):
    return mesh.sample_points_poisson_disk(n_points)

pixie_top_pcd    = mesh_to_pcd(pixie_top)
pixie_bottom_pcd = mesh_to_pcd(pixie_bottom)
alpha_top_pcd    = mesh_to_pcd(alpha_top)
alpha_bottom_pcd = mesh_to_pcd(alpha_bottom)


def part_icp(source_pcd, target_pcd, voxel=0.015, max_corr_dist_init=0.08):
    source_down, source_f = preprocess(source_pcd, voxel)
    target_down, target_f = preprocess(target_pcd, voxel)

    # Global registration (RANSAC)
    result_ransac = o3d.pipelines.registration.registration_ransac_based_on_feature_matching(
        source_down, target_down, source_f, target_f,
        mutual_filter=False,
        max_correspondence_distance=max_corr_dist_init,
        estimation_method=o3d.pipelines.registration.TransformationEstimationPointToPoint(False),
        ransac_n=4,
        checkers=[
            o3d.pipelines.registration.CorrespondenceCheckerBasedOnEdgeLength(0.9),
            o3d.pipelines.registration.CorrespondenceCheckerBasedOnDistance(max_corr_dist_init)
        ],
        criteria=o3d.pipelines.registration.RANSACConvergenceCriteria(max_iteration=200000, confidence=0.999)
    )

    # ICP refinement (multi-step, after RANSAC)
    result_icp_1 = o3d.pipelines.registration.registration_icp(
        source_pcd, target_pcd,
        max_correspondence_distance=max_corr_dist_init,
        init=result_ransac.transformation,
        estimation_method=o3d.pipelines.registration.TransformationEstimationPointToPlane(),
        criteria=o3d.pipelines.registration.ICPConvergenceCriteria(max_iteration=100)
    )

    result_icp_2 = o3d.pipelines.registration.registration_icp(
        source_pcd, target_pcd,
        max_correspondence_distance=max_corr_dist_init/2,
        init=result_icp_1.transformation,
        estimation_method=o3d.pipelines.registration.TransformationEstimationPointToPlane(),
        criteria=o3d.pipelines.registration.ICPConvergenceCriteria(max_iteration=200)
    )

    result_icp_3 = o3d.pipelines.registration.registration_icp(
        source_pcd, target_pcd,
        max_correspondence_distance=max_corr_dist_init/5,
        init=result_icp_2.transformation,
        estimation_method=o3d.pipelines.registration.TransformationEstimationPointToPlane(),
        criteria=o3d.pipelines.registration.ICPConvergenceCriteria(max_iteration=300)
    )

    return result_icp_3.transformation

# o3d.visualization.draw_geometries(
#     [alpha_bottom_pcd, alpha_top_pcd, pixie_bottom_pcd, pixie_top_pcd],
#     mesh_show_back_face=True
# )

trans_top    = part_icp(pixie_top_pcd, alpha_top_pcd)
trans_bottom = part_icp(pixie_bottom_pcd, alpha_bottom_pcd)

# Apply transformations to original meshes
pixie_top.transform(trans_top)
pixie_bottom.transform(trans_bottom)

# Merge top + bottom back together
pixie_combined = pixie_top + pixie_bottom

o3d.visualization.draw_geometries(
    [alpha_mesh, pixie_combined],
    mesh_show_back_face=True
)


[Open3D WARNING] [RemoveDuplicatedTriangles] This mesh contains triangle uvs that are not handled in this function
[Open3D WARNING] [RemoveDegenerateTriangles] This mesh contains triangle uvs that are not handled in this function
[Open3D WARNING] [RemoveDuplicatedTriangles] This mesh contains triangle uvs that are not handled in this function
[Open3D WARNING] [RemoveDegenerateTriangles] This mesh contains triangle uvs that are not handled in this function


In [129]:
# clean pixie bottom, so alpha is on top
pixie_vertices = np.asarray(pixie_bottom.vertices)

# Sample Alpha vertices and build a KDTree for depth check
alpha_pcd = alpha_bottom.sample_points_poisson_disk(50000)
alpha_tree = o3d.geometry.KDTreeFlann(alpha_pcd)

# Keep only Pixie vertices that are behind Alpha in Z
keep_idx = []
for i, v in enumerate(pixie_vertices):
    _, idx, dists = alpha_tree.search_knn_vector_3d(v, 1)
    closest_alpha = np.asarray(alpha_pcd.points)[idx[0]]
    if v[2] > closest_alpha[2]:  # Pixie behind Alpha
        keep_idx.append(i)

pixie_back_only = pixie_bottom.select_by_index(keep_idx, cleanup=True)

# Combine with Alpha (grey always on top)
combined_bottom = alpha_bottom + pixie_back_only
combined_bottom.remove_duplicated_vertices()
combined_bottom.remove_duplicated_triangles()
combined_bottom.compute_vertex_normals()
combined_bottom.paint_uniform_color([0.7, 0.7, 1.0])

o3d.visualization.draw_geometries([combined_bottom], mesh_show_back_face=True)


pcd = o3d.geometry.PointCloud()
pcd.points = combined_bottom.vertices
pcd.normals = combined_bottom.vertex_normals  # normals help Poisson a lot

# Poisson reconstruction (fills holes, smooths mesh)
mesh_poisson, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(
    pcd, depth=12
)

# Optional: crop to original bounding box to remove far-away artifacts
bbox = combined_bottom.get_axis_aligned_bounding_box()
mesh_poisson = mesh_poisson.crop(bbox)

# Compute normals again for good measure
mesh_poisson = mesh_poisson.filter_smooth_simple(number_of_iterations=2)
mesh_poisson.compute_vertex_normals()

# Visualize
o3d.visualization.draw_geometries([mesh_poisson])

[Open3D WARNING] [RemoveDuplicatedTriangles] This mesh contains triangle uvs that are not handled in this function
[Open3D WARNING] [RemoveDegenerateTriangles] This mesh contains triangle uvs that are not handled in this function


In [121]:
# clean pixie top, so alpha is on top
pixie_vertices = np.asarray(pixie_top.vertices)

# Sample Alpha vertices and build a KDTree for depth check
alpha_pcd = alpha_top.sample_points_poisson_disk(50000)
alpha_tree = o3d.geometry.KDTreeFlann(alpha_pcd)

# Keep only Pixie vertices that are behind Alpha in Z
keep_idx = []
for i, v in enumerate(pixie_vertices):
    _, idx, dists = alpha_tree.search_knn_vector_3d(v, 1)
    closest_alpha = np.asarray(alpha_pcd.points)[idx[0]]
    if v[2] > closest_alpha[2]:  # Pixie behind Alpha
        keep_idx.append(i)

pixie_back_only = pixie_top.select_by_index(keep_idx, cleanup=True)

# Combine with Alpha (grey always on top)
combined_top = alpha_top + pixie_back_only
combined_top.remove_duplicated_vertices()
combined_top.remove_duplicated_triangles()
combined_top.compute_vertex_normals()
combined_top.paint_uniform_color([1.0, 0.7, 0.7])

# Visualize
o3d.visualization.draw_geometries([combined_top], mesh_show_back_face=True)

pcd = o3d.geometry.PointCloud()
pcd.points = combined_top.vertices
pcd.normals = combined_top.vertex_normals  # normals help Poisson a lot
# Poisson reconstruction (fills holes, smooths mesh)
mesh_poisson, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(
    pcd, depth=16
)

# Optional: crop to original bounding box to remove far-away artifacts
bbox = combined_top.get_axis_aligned_bounding_box()
mesh_poisson = mesh_poisson.crop(bbox)

# Compute normals again for good measure
mesh_poisson.compute_vertex_normals()

# Visualize
o3d.visualization.draw_geometries([mesh_poisson])

[Open3D WARNING] [RemoveDuplicatedTriangles] This mesh contains triangle uvs that are not handled in this function
[Open3D WARNING] [RemoveDegenerateTriangles] This mesh contains triangle uvs that are not handled in this function


[WARNING] /Users/runner/work/Open3D/Open3D/build/poisson/src/ext_poisson/PoissonRecon/Src/FEMTree.Initialize.inl (Line 193)
          Initialize
          Found bad data: 4


Step 2: Identify PIXIE mesh parts to remove


In [122]:
pixie_normals = np.asarray(pixie_combined.vertex_normals)
pixie_vertices_full = np.asarray(pixie_combined.vertices)

head_mask_pixie = pixie_vertices_full[:,1] > (
    pixie_vertices_full[:,1].min() + 0.85 * (pixie_vertices_full[:,1].ptp())
)

# Arms = x far from center
x_min, x_max = pixie_vertices_full[:,0].min(), pixie_vertices_full[:,0].max()
x_center = (x_min + x_max) / 2
arm_thresh = 0.25 * (x_max - x_min)

arm_mask_pixie = (pixie_vertices_full[:,0] < x_center - arm_thresh) | \
                 (pixie_vertices_full[:,0] > x_center + arm_thresh)

# Keep head + arms from PIXIE
pixie_head_arms_mask = head_mask_pixie | arm_mask_pixie
pixie_head_arms = pixie_combined.select_by_index(
    np.where(pixie_head_arms_mask)[0].tolist(),
    cleanup=True
)

pixie_normals = np.asarray(pixie_combined.vertex_normals)
back_mask = pixie_normals[:,2] >= -0.25   # normals NOT facing camera

pixie_back_mask = back_mask & (~pixie_head_arms_mask)
pixie_back_only = pixie_combined.select_by_index(
    np.where(pixie_back_mask)[0].tolist(),
    cleanup=True
)

depth_verts = np.asarray(alpha_mesh.vertices)

head_mask_depth = depth_verts[:,1] > (
    depth_verts[:,1].min() + 0.80 * depth_verts[:,1].ptp()
)

x_min, x_max = depth_verts[:,0].min(), depth_verts[:,0].max()
x_center = (x_min + x_max) / 2
arm_thresh = 0.25 * (x_max - x_min)
arm_mask_depth = (depth_verts[:,0] < x_center - arm_thresh) | \
                 (depth_verts[:,0] > x_center + arm_thresh)

remove_depth_mask = head_mask_depth | arm_mask_depth
keep_depth_mask = ~remove_depth_mask

depth_filtered = alpha_mesh.select_by_index(
    np.where(keep_depth_mask)[0].tolist(),
    cleanup=True
)

alpha_pcd = depth_filtered.sample_points_poisson_disk(50000)
tree = o3d.geometry.KDTreeFlann(alpha_pcd)

pixie_back_vertices = np.asarray(pixie_back_only.vertices)
keep = []

for i, v in enumerate(pixie_back_vertices):
    _, idx, dist = tree.search_knn_vector_3d(v, 1)
    if dist[0] > 0.0001:
        keep.append(i)

pixie_back_pruned = pixie_back_only.select_by_index(keep, cleanup=True)

combined = pixie_head_arms + pixie_back_pruned + depth_filtered
combined.remove_duplicated_vertices()
combined.remove_duplicated_triangles()
combined.remove_non_manifold_edges()
combined.compute_vertex_normals()

o3d.visualization.draw_geometries(
    [combined],
    mesh_show_back_face=True
)


[Open3D WARNING] [RemoveDuplicatedTriangles] This mesh contains triangle uvs that are not handled in this function
[Open3D WARNING] [RemoveDegenerateTriangles] This mesh contains triangle uvs that are not handled in this function
[Open3D WARNING] [RemoveDuplicatedTriangles] This mesh contains triangle uvs that are not handled in this function
[Open3D WARNING] [RemoveDegenerateTriangles] This mesh contains triangle uvs that are not handled in this function


KeyboardInterrupt: 

Non-rigid transformations

In [ ]:
# # read in the meshes
# alpha_mesh = o3d.io.read_triangle_mesh("/Users/adeleyounis/Desktop/Capstone/wAI/3D-processing/alpha_mesh.obj")
# alpha_mesh.compute_vertex_normals()

# pixie_mesh = o3d.io.read_triangle_mesh("/Users/adeleyounis/Desktop/Capstone/wAI/3D-processing/modelling/output/adele_down/adele_down.obj")
# pixie_mesh.compute_vertex_normals()

# alpha_mesh.paint_uniform_color([0.7, 0.7, 0.7])
# pixie_mesh.paint_uniform_color([1.0, 0.2, 0.2])

# # verify that they are triangular meshes with points and triangles
# print(f"alpha_mesh: {alpha_mesh}")
# print(f"pixie_mesh: {pixie_mesh}")

In [ ]:
# def mesh_to_points(mesh, n=20000):
#     pcd = mesh.sample_points_uniformly(number_of_points=n)
#     return np.asarray(pcd.points)

# # 1) sample points (use fewer first to tune)
# X = mesh_to_points(pixie_mesh, n=15000)  # source (will deform)
# Y = mesh_to_points(alpha_mesh, n=15000)  # target

# # 2) run CPD non-rigid
# reg = DeformableRegistration(X=X, Y=Y, beta=2.0, lamb=3.0, max_iterations=80, tol=1e-5)
# TY, _ = reg.register()  # TY is X deformed toward Y (same shape as X)

# # 3) warp the SOURCE mesh vertices using the CPD deformation learned on sampled points
# V = np.asarray(alpha_mesh.vertices)
# tree = cKDTree(X)
# _, idx = tree.query(V, k=1)
# V_warped = TY[idx]

# alpha_warped = o3d.geometry.TriangleMesh(
#     vertices=o3d.utility.Vector3dVector(V_warped),
#     triangles=alpha_mesh.triangles
# )
# alpha_warped.compute_vertex_normals()
# alpha_warped.paint_uniform_color([0.2, 0.8, 0.2])

# o3d.visualization.draw_geometries([pixie_mesh, alpha_warped])
